# Experiment v6 — one notebook, both phases (Dell PowerEdge R7615)

v6 asks the same question twice, in two ways that answer different halves of it:

| | phase | what it measures | output tree |
|---|---|---|---|
| **A** | **isolation** | the value of the best sequence found in ONE turn from a captured position, under a fixed compute budget — *search quality*, with full telemetry | `test_results/v6` |
| **B** | **full games** | who wins complete games at those same budgets — *playing strength*, and nothing else | `test_results/v6_full` |

Neither substitutes for the other: a better sequence in one turn need not become a win, and a win rate
cannot say which operator produced it. They share a machine, a build, a compute-unit currency and the
three budget levels, which is why they now share a notebook.

**Run order.** Part 0 once (setup + machine calibration), then Part A, then Part B. Within a part, run
the cells in order — Part A's §A4 (C0) decides the budget levels and Part B's §B3 decides the turn cap,
and both come before their phase's run cell.

**The two phases must never run at the same time.** Separate trees and separate locks, so nothing
would be corrupted — but they would compete for the same cores, and every throughput number either one
measured would be meaningless.

What is new in v6:
- **Greedy initial sequence** (standard EMCTS's default), shared across all 12 configs per (scenario, visit).
- **The budget bounds SEARCH only** — greedy initialization is measured and excluded; bridge burning,
  including the crossover tournament, stays inside the budget.
- **Compute-units are calibrated per configuration and per budget level** (§0.5). An advance, a copy
  and an eval do not cost the same for every algorithm or at every budget, so one global weight set
  handed some agents more real compute than others for the same nominal budget. Each configuration is
  now priced by its own measured weights.
- **100 scenarios** (v5: 200) at **50 visits** each (v5: 100).
- **Realized spend logged per turn** (units, advances, copies, evals, init cost, wall-clock).

How this notebook treats the server (see `analysis/V6_SERVER_RUNBOOK.md` for the full rationale):
- **Threads are capped at PHYSICAL cores, never the SMT count.** `ServerProfile` reads
  `/proc/cpuinfo`, subtracts a small reserve for GC/JIT/OS, and that ceiling binds every pool in the
  pipeline — portfolio build, thread calibration, isolation phase and tournament alike.
- **The thread count below that ceiling is measured, not guessed**, per budget level, from *sustained*
  throughput — each sweep point runs twice so a machine that throttles under load is visible.
- **A load governor lowers concurrency during a run** if the heap comes under pressure, and raises it
  again when it clears.
- **Every stage is restartable.** Ctrl-C / SIGTERM drains cleanly; a hard kill costs at most one turn
  (Part A) or one game (Part B) per worker. Re-running the same cell resumes.
- **Only one writer per output tree.** Each job takes a heartbeat lock and refuses to start beside a
  live one, so "re-run the cell to resume" can never silently double-write a dataset.

In [ ]:
# EDIT to the absolute path of the Tribes-master directory on the server:
TRIBES = '/root/Bachelor_Thesis/Tribes-master'
%cd $TRIBES

## 0.1 Environment check
Java, CPU topology, memory, disk. Three things to confirm before anything else:

1. **Physical cores vs threads.** `lscpu` reports `Core(s) per socket` × `Socket(s)` as the physical
   count and `Thread(s) per core` as SMT. On a fully-populated R7615 that is 96 physical / 192 logical.
   The pipeline will use the physical number minus a reserve, and says so in its own banner.
2. **Nothing else is running.** §0.5 (compute weights) and §A4 (C0) are *timing* experiments on this
   exact machine — competing load makes their numbers wrong rather than merely noisy.
3. **No v6 job already holds a lock**, in either tree. A lock listed below with a live PID means a run
   is still going; a lock from a `kill -9` goes stale on its own within two minutes.

In [ ]:
!java -version 2>&1 | head -3
print('--- CPU topology ---')
!lscpu | egrep 'Model name|^CPU\(s\)|Thread\(s\) per core|Core\(s\) per socket|Socket\(s\)|NUMA node\(s\)|MHz'
print('--- memory / disk ---')
!free -g | head -2
!df -h . | tail -1
print('--- power/thermal governor (affects sustained clocks) ---')
!cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_governor 2>/dev/null || echo '(cpufreq not exposed)'
print('--- other java processes (should be none before a timing run) ---')
!pgrep -af java || echo '(none - good)'
print('--- v6 run locks currently held (both phases) ---')
!find test_results/v6 test_results/v6_full -name '_running.lock' -exec sh -c 'echo "$1: $(cat $1)"' _ {} \; 2>/dev/null || true

## 0.2 Build
Recompiles everything from `src/`. The source list is regenerated each time, so new classes
(`ServerProfile`, `RunGuard`, `LoadGovernor`, the rewritten `ThreadCalibrationV6`, `BuildV6Portfolio`,
and the whole full-games phase) are picked up automatically.

One change here touches the shared engine: `Game.printGameResults()` is gated on `Constants.VERBOSE`.
Interactive play is unchanged (it defaults to true); the tournament sets it false so 3,300 games across
~90 threads do not interleave five lines each into the run log.

In [ ]:
!rm -rf out && mkdir -p out
!find src -name '*.java' > _srcs.txt
# --release 11 pins the language level AND the API set to the server's JDK, so anything newer is a
# compile error on the developer's machine rather than a surprise here. (It already caught one:
# Thread.stop() is final up to JDK 22 and removed in 23, so a HeapSampler.stop() compiled fine on a
# JDK 26 laptop and failed on this server.)
!javac --release 11 -nowarn -d out -cp "lib/*" @_srcs.txt && echo 'BUILD OK' || echo 'BUILD FAILED'
!echo "classes: $(find out -name '*.class' | wc -l)"

## 0.3 Machine policy — what the pipeline will actually do
Prints the ceiling and the thread ladder this machine gets, without running anything. Check it against
§0.1 before committing to a multi-day run.

To override (a shared machine, or an explicit core count when `/proc/cpuinfo` is unhelpful):

- `-Dv6.maxThreads=N`  — hard cap on workers for every stage
- `-Dv6.reserveCores=N` — cores held back for GC/JIT/OS (default ~3%, min 1, max 4)

Both are also readable from the environment as `V6_MAX_THREADS` / `V6_RESERVE_CORES`.

In [ ]:
!java -cp "out:lib/*" -Xmx4g core.data.exp.ServerProfile

## 0.4 Portfolio (~30 min, parallel + restartable)
Builds the **100-scenario** portfolio (50 single + 50 dual). It is Part A's experimental material, but
it lives in the shared setup because **§0.5 measures on it**: per-operation costs depend on the states
being searched, and a calibration done on fresh-game states would not transfer to the positions the
experiment actually plans from.

Source games run continuously across workers and each finished game is journalled, so **an interrupted
build resumes instead of replaying every game from attempt 1** — re-run this cell after any
interruption.

Counts come from `BuildV6Portfolio.NUM_SINGLE_SCENARIOS` / `NUM_DUAL_SCENARIOS`; the two optional
trailing arguments override them. A completed portfolio records the counts it was built for, so
changing them forces a rebuild rather than silently reusing the wrong set.

The build takes the machine's worker ceiling automatically; pass a number instead of `auto` to force
one. Heap: source games are small, 64 GB is generous.

In [ ]:
!java -Xmx64g -cp "out:lib/*" core.data.exp.BuildV6Portfolio test_results/v6/_portfolio auto

## 0.5 Compute-unit weights — per configuration, per budget level (~35 min)

A compute-unit is `W_ADVANCE*advances + W_COPY*copies + W_EVAL*evals`. Those weights are the relative
wall-clock costs of the three operations — a property of the **machine**, not of the code.

**They are also a property of the configuration and of the budget level, which is why this cell is no
longer a three-minute job.** A crossover agent splices sequences and repairs them, so the states it
advances, copies and evaluates are not the states a mutation-only agent reaches; and at ten times the
budget the tree stops fitting in cache, so every operation costs more than it did at the floor. (That
second effect is already in the record: the full-games plan measured the 400,000-unit level at **26.7×**
the wall-clock of the 40,000-unit level rather than 10×.) Priced with one weight set, a nominal budget
of N units buys measurably different amounts of real compute per agent and per level — the exact
confound the budget exists to remove.

So this measures all **12 configs × 3 levels = 36 cells**, on the same scenarios with the same seeds,
and writes a weight set for each.

**What a unit means afterwards.** Every weight is relative to one reference operation: an advance in
`EMCTS@40000`. So `W_ADVANCE` is 1.0 only for the reference — an advance that genuinely takes 1.4× as
long is charged 1.4 units. Consequences worth knowing before you read any result:

1. equal unit budgets are equal wall-clock **across configurations** — the fairness property the budget
   was introduced for, now measured rather than assumed;
2. the levels become proportional **in time**: 400,000 units costs ~10× 40,000 units instead of 26.7×,
   because the extra per-op cost is now charged rather than absorbed. The tournament in Part B gets
   correspondingly cheaper;
3. a unit has a stated nanosecond value on this machine, written into the file.

Read the **DOES AN OPERATION COST THE SAME FOR EVERY CONFIGURATION?** block: it is the evidence that
per-cell weights were needed, and the number to quote in the write-up.

The **IS THIS A FAIR BENCHMARK?** block is the gate. Per cell: enough operations to divide by, CV
across repetitions ≤ 10%, plausible weights; plus an idle machine overall. If anything fails it writes
**no** weights (the measurements still land in `compute_profiles.csv` for inspection), so a bad
measurement can never silently reprice the experiment. The usual cause is other load: stop it and
re-run.

`ComputeMeter` loads `test_results/v6/_calibration/compute_weights.properties` on startup wherever it
runs, so **both phases** pick this up with nothing to copy and nothing to rebuild.

**Do this before §A4 (C0) and before either run.** Changing the weights changes what a unit *is*, so
the budget levels have to be re-derived afterwards (§A3) — and never change them mid-experiment.

Args: `[portfolioDir] [outDir] [reps] [turnsPerRepCsv] [warmupTurns] [levelsCsv]`.

In [ ]:
!java -Xmx32g -cp "out:lib/*" core.data.exp.ComputeCalibrateV6

In [ ]:
# What is now in force. Re-run any time to confirm.
import os
import pandas as pd

CALIB = 'test_results/v6/_calibration'
prof = f'{CALIB}/compute_profiles.csv'
if not os.path.exists(prof):
    print('no calibration yet - every config would fall back to the compiled global weights')
else:
    p = pd.read_csv(prof)
    display(p[['Profile', 'BudgetUnits', 'AdvanceNanos', 'CopyNanos', 'EvalNanos',
               'W_ADVANCE', 'W_COPY', 'W_EVAL', 'Status']].round(4))

    # The headline: how far apart the configs are within a level, and what the old single
    # weight set would have mis-allocated between them.
    print('\nper-advance cost spread within each level (this is why the weights are per-config):')
    g = p.groupby('BudgetUnits').AdvanceNanos.agg(['min', 'max'])
    g['spread_pct'] = (100 * (g['max'] / g['min'] - 1)).round(1)
    display(g.round(1))

    print('what 1,000 units really bought under the OLD global weights (ms), by level:')
    o = p.groupby('BudgetUnits').MsPerKiloUnitOld.agg(['min', 'max'])
    o['unfairness_pct'] = (100 * (o['max'] / o['min'] - 1)).round(1)
    display(o.round(3))
    print('Under the new per-config weights that column is constant by construction '
          f"({p.MsPerKiloUnitNew.iloc[0]:.2f} ms per 1,000 units) - that constancy IS the fix.")

    bad = p[p.Status != 'ok']
    print(f'\ncells with problems: {len(bad)}' + ('' if len(bad) == 0 else ' <-- weights were NOT written'))
    if len(bad):
        display(bad[['Profile', 'Status']])

---
# PART A — Isolation phase (`test_results/v6`)

One search turn per game, from a captured position, under a fixed compute-unit budget. Every
configuration sees the same scenarios, the same visit seeds and the same greedy root, so the
comparison is paired all the way down.

Scale: **100 scenarios × 50 visits × 12 configs × 3 budget levels = 180,000 turns** (v5: 720,000).
Each turn is ~3× v5's budget, so plan for roughly 1.5× v5's isolation-phase wall-clock.

The portfolio is half of v5's — 50 single-source + 50 dual-source scenarios. The cut is taken from
scenarios rather than from visits on purpose: the independent replicate in the analysis is the
*scenario* (the 50 visits of one scenario share a board, a position and a greedy root, and
`v6_significance.py` averages them before testing), so halving visits would have bought the same
wall-clock while hiding the cost, whereas halving scenarios states it — n goes 200 → 100, about a
√2 widening of the paired test's resolution.

## A1. Smoke test (~30 s) — pipeline correctness
Verifies FastGameStateIO round-trip, the shared **greedy** initial sequence across agents AND budgets,
folder layout, CSV schema, operator telemetry, the two budget-boundary properties, thread invariance,
and the per-config weight profiles:

- **init OUT of the budget**: every agent reports a non-zero initialization cost while its search spend
  still reaches the full budget (if init were charged, search would fall short by exactly that amount);
- **bridge burning IN the budget**: all agents perform 20 burns, the crossover agents stay at the budget
  rather than exceeding it, and they buy strictly fewer distinct sequences than EMCTS — the tournaments
  are paid for out of the same pot;
- **§9 weight profiles**: every configuration asks for the key the calibration writes, pricing follows
  the active weights, a profile cannot leak from one turn to the next on a pooled worker thread, and —
  now that §0.5 has run — all 36 expected profiles are present. Every one of those failures is
  otherwise silent: the run completes and is priced wrong.

If this fails, do NOT launch C0 or the full run.

In [ ]:
!java -Xmx4g -cp "out:lib/*" core.data.exp.NewExperimentSmokeTestV6

## A2. Is the budget fair between the agents, and does each operator fire?
`BudgetFairnessProbe` measures each configuration's realized operation mix, prints which weight profile
priced each one, and re-prices all of them under alternative weights.

Read the spread it reports as **the size of the bias the per-config profiles remove**: it is how far
apart the configurations move when a single weight is wrong, and with §0.5's profiles in force that
distortion is being corrected rather than carried. If any configuration shows up as NOT CALIBRATED, it
silently fell back to the global weights and the distortion is live — re-run §0.5 before continuing.

It also reports budget adherence and whether each crossover operator actually fires at that budget.

In [ ]:
# Read the VERDICT block of each. Budgets come from RunExperimentV6.DEFAULT_COMPUTE_LEVELS.
LEVELS = (40000, 80000, 400000)
for cu in LEVELS:
    print()
    print('=' * 70)
    print('  budget', cu, 'cu')
    print('=' * 70)
    !java -Xmx32g -cp "out:lib/*" core.data.exp.BudgetFairnessProbe test_results/v6/_portfolio test_results/v6/_calibration {cu} 12

## A3. Budget floor — why the smallest level is what it is
The crossover bridge-burn tournament is a 3-level bracket over the phase's top **eight** elites
(8 → 4 → 2 → 1). The code runs it whenever the pool holds at least **two**, so below a certain budget a
burn either skips the tournament entirely — falling back to promoting the best child, which is what
plain EMCTS does — or breeds from a pool of 2 or 3 and is not the operator being studied. Neither
announces itself in the results.

This probe sweeps candidate budgets and reports, for **CrossoverBB and CrossoverCombo separately**
(BB is the binding case — it has no expansion crossover feeding its pool), the share of the 20 burns
that ran a real tournament, the mean bracket size out of 8, and how many new nodes are added per phase
between consecutive burns. Re-run it if the compute weights change, since a unit will mean something
different.

In [ ]:
!java -Xmx16g -cp "out:lib/*" core.data.exp.BudgetFloorProbe test_results/v6/_portfolio test_results/v6/_calibration 26800,33500,40000,47000,60000 10

## A4. C0 — thinking-time calibration (~2.6 h, SINGLE-THREADED)
**Run this before §A5 and read its output.** Gives each of the 12 configs a fixed amount of SEARCH time
(500 / 1000 / 5000 ms, each a separate run) and records what it actually does: advances, copies, evals,
weighted units, and the greedy-initialization cost separately.

It answers four questions:
1. the units→seconds conversion the paper reports for this reference machine;
2. **whether §0.5's per-config weights actually equalised the currency** — units/ms should now be FLAT
   across the 12 configs. This used to be an open question ("is one shared budget fair, or do we need
   per-agent budgets?"); it is now a check on work already done, and the sharpest one available,
   because C0 is the only v6 job that holds time fixed and lets the work vary;
3. whether it is invariant across scenario source and branching factor (the H5 confound check);
4. how large `InitUnits` actually is, and therefore whether the smallest budget level needs raising.

Its units are priced by the profile of ONE budget level (the seventh argument, default 40,000), which
is recorded in the `ProfileLevel` column. A target far from that level's scale will drift, because
per-op cost grows with tree size — the same effect the per-level profiles exist to price, seen from the
other side.

It samples 12 scenarios per source rather than the whole portfolio, so its cost did not change when the
portfolio was halved.

Single-threaded **on purpose**: this machine is the reference machine, so the run must be the only load
on it. Leave it otherwise idle — do not start §A5 alongside it.

Interrupted? Re-run the same cell: fully-measured targets are kept and only the interrupted one is
redone. A target that is only half-measured is discarded rather than mixed with a second run's. A CSV
written before the `ProfileLevel` column existed is discarded rather than mixed, too.

In [ ]:
# args: [portfolioDir] [outDir] [targetsMsCsv] [perSource] [reps] [warmup] [profileUnits]
!nohup java -Xmx32g -cp "out:lib/*" core.data.exp.CalibrationRunnerV6 \
    test_results/v6/_portfolio test_results/v6/_calibration 500,1000,5000 12 5 40 40000 \
    > v6_calibration.log 2>&1 &
print('C0 started in background -> v6_calibration.log')

In [ ]:
# C0 progress + the per-config summary once it finishes (re-run to refresh).
!echo "rows so far : $(($(wc -l < test_results/v6/_calibration/calibration.csv 2>/dev/null || echo 1)-1))"
!pgrep -af CalibrationRunnerV6 || echo '(C0 not running - finished or not started)'
!tail -n 40 v6_calibration.log 2>/dev/null || echo '(no log yet)'

### A4b. Read C0 → verify the currency, then set the budget levels
The first table is the **verification of §0.5**. Read *units/ms across configs at a fixed target*: with
per-config weights in force it should be flat, because a unit is defined as a fixed amount of
wall-clock. A configuration sitting 10% above the others is getting 10% more real compute than its
nominal budget says — the weights are stale, were measured on another machine, or that config's
profile is missing (§A2 prints which profile priced each one).

The `flatness` line below turns that into one number: the spread between the fastest and slowest
config's units/ms, per target. Under the old single weight set this was ~13–18% on the dev laptop.

Then check `InitUnits` against the smallest level (40,000): initialization is excluded from the budget,
but if it is a large fraction of it, the turn is spending more outside the measured budget than inside
it, which is worth reporting.

Note the smallest level is set by `BudgetFloorProbe` (§A3), not by C0 — C0 answers "what does an agent
do in T ms", which is a different question from "is the operator structurally able to run".

In [ ]:
import pandas as pd
c = pd.read_csv('test_results/v6/_calibration/calibration.csv')
c['Config'] = c.Config + c.Ratio.astype(str).radd('@r').where(c.Ratio != 'NA', '')
c['UnitsPerMs'] = c.SearchUnits / c.SearchMillis.clip(lower=1)
print('units priced by the', sorted(c.ProfileLevel.unique()), 'cu weight profile(s)')

print('=== units/ms by config x target -- FLAT is the pass condition ===')
piv = c.pivot_table(index='Config', columns='TargetMillis', values='UnitsPerMs', aggfunc='mean')
display(piv.round(2))

print('flatness (max/min - 1) per target; small = the per-config weights did their job:')
flat = ((piv.max() / piv.min() - 1) * 100).round(1)
for t, v in flat.items():
    verdict = 'flat' if v < 5 else ('drifting' if v < 15 else 'NOT FLAT - investigate')
    print(f'  {t:>6} ms target: {v:5.1f}%  {verdict}')
print('  (worst config at each target:')
for t in piv.columns:
    print(f'    {t:>6} ms: high={piv[t].idxmax()} {piv[t].max():.2f}  '
          f'low={piv[t].idxmin()} {piv[t].min():.2f}')
print('  )')

print('=== units/ms by scenario source (the H5 confound check) ===')
display(c.pivot_table(index='Source', columns='TargetMillis', values='UnitsPerMs', aggfunc='mean').round(2))

print('=== greedy-init cost by source, and as a share of the smallest planned level ===')
init = c.groupby('Source').InitUnits.agg(['mean', 'median', 'max']).round(0)
init['share_of_floor'] = (init['mean'] / 40000).round(3)   # vs the smallest v6 level
display(init)

print('=== stability across repetitions (CV should be small on a quiesced machine) ===')
cv = c.groupby(['TargetMillis', 'Config']).SearchUnits.agg(['mean', 'std'])
cv['cv'] = (cv['std'] / cv['mean']).round(4)
display(cv.cv.groupby('TargetMillis').describe()[['mean', 'max']].round(4))

print('=== units actually spent per config x target (should also be flat) ===')
display(c.pivot_table(index='Config', columns='TargetMillis', values='SearchUnits', aggfunc='mean').round(0))
print('Per-AGENT budgets are no longer the fallback if this is uneven: the weights are what is')
print('calibrated per agent now, so unevenness here means re-run section 0.5, not re-plan the design.')

## A5. Run the isolation phase (background + log) — also the RESUME cell
Launches the run detached so it survives a disconnect. **Re-run this same cell to resume** after a
server restart — finished work is skipped, an interrupted config continues per (scenario, visit), and
the run refuses to start if a previous one is still alive.

Args: `[threads|auto] [computeUnitsCsv] [numSingle] [numDual]`. `auto` (default) runs the pre-run
phases: portfolio build → startup verify → **greedy-init invariance check** → **per-budget thread
calibration** → isolation.

The budget levels default to **40,000 / 80,000 / 400,000** (floor, 2×, 10×), derived by
`BudgetFloorProbe` in §A3 — see `V6_SERVER_RUNBOOK.md` §5d. Override by passing them explicitly, e.g.
`auto 40000,80000,400000`. If C0 or a re-calibration of the compute weights moves the floor, re-run
§A3 and pass the new set.

Check the banner line **`Compute profiles: 36/36 calibrated`** in the first lines of `v6_run.log`.
Anything less means some configurations are being priced with the global weights instead of their own,
and the budget is not equal compute across them — stop, run §0.5, and start again.

Heap: give the JVM the machine's RAM minus what the OS needs. `-Xmx750g` suits a 768 GB R7615; lower it
and the thread calibration will simply choose fewer workers. `-XX:+UseG1GC` and a generous
`-XX:MaxGCPauseMillis` suit a throughput job with large short-lived trees; the load governor reads G1's
post-collection occupancy to decide when to throttle.

In [ ]:
!nohup java -Xmx750g -XX:+UseG1GC -XX:MaxGCPauseMillis=500 \
    -cp "out:lib/*" core.data.exp.RunExperimentV6 auto > v6_run.log 2>&1 &
print('v6 isolation phase started/resumed in background -> v6_run.log')

### A5b. Progress (re-run to refresh)
Read-only view of the running job: pre-run checkpoints, the thread count chosen per budget level,
leaves finished, turns/min with a projected finish, live thread/CPU use (`NLWP` = OS threads, `%CPU` =
total across them), and the log tail. Governor lines in the tail show any throttling.

In [ ]:
import subprocess, time, os, glob, re

BASE, LEAVES = 'test_results/v6', 36
SCENARIOS, VISITS = 100, 50              # BuildV6Portfolio counts x IsolationPhaseRunnerV6.VISITS_PER_SCENARIO
TURNS_PER_LEAF = SCENARIOS * VISITS

!echo "cores       : $(nproc) logical / $(lscpu | awk -F: '/^Core\(s\) per socket/{c=$2} /^Socket\(s\)/{s=$2} END{print c*s}') physical"
!echo "portfolio   : $(test -f test_results/v6/_portfolio/_portfolio_complete.txt && echo COMPLETE || echo building/incomplete)"
!echo "greedy-init : $(test -f test_results/v6/_greedy_init_ok.txt && echo ok || echo pending)"
!echo "calibration : $(test -f test_results/v6/_calibration/calibration.csv && echo present || echo 'MISSING - run section A4 first')"
!echo "lock        : $(cat test_results/v6/_running.lock 2>/dev/null || echo '(none - no run in progress)')"

# Thread count chosen per budget level (one file per level).
tuned = sorted(glob.glob(f'{BASE}/_tuned_threads_*cu.txt'))
if tuned:
    picks = ', '.join(f"{re.search(r'_([0-9]+)cu', f).group(1)}cu -> {open(f).read().strip()} threads"
                      for f in tuned)
    print('threads     :', picks)
else:
    print('threads     : pending (calibration has not finished a level yet)')

# Turn counts per leaf, from the telemetry file (one row per completed turn).
teles = sorted(glob.glob(f'{BASE}/**/turn_telemetry.csv', recursive=True))
total_turns = 0
print(f'--- per-leaf completed turns (full leaf = {TURNS_PER_LEAF}) ---')
for f in teles:
    n = max(0, sum(1 for _ in open(f)) - 1)
    total_turns += n
    if n < TURNS_PER_LEAF:
        print(f'  {f.replace(BASE + "/", ""):50s} {n:6d}')
print(f'  TOTAL {total_turns} / {LEAVES * TURNS_PER_LEAF} turns '
      f'({100.0 * total_turns / (LEAVES * TURNS_PER_LEAF):.1f}%)')

# Throughput + ETA from the log's age.
log = 'v6_run.log'
if os.path.exists(log) and total_turns > 0:
    mins = max(1e-9, (time.time() - os.path.getctime(log)) / 60.0)
    rate = total_turns / mins
    left = LEAVES * TURNS_PER_LEAF - total_turns
    if rate > 0:
        print(f'  rate  {rate:.0f} turns/min   elapsed {mins/60:.1f} h   ETA {left / rate / 60:.1f} h')

print('--- run process: PID / threads (NLWP) / CPU% / elapsed ---')
!ps -o pid,nlwp,pcpu,etime,comm -C java 2>/dev/null || echo '(no java process running)'
!pgrep -af RunExperimentV6 || echo '(RunExperimentV6 not running - finished, not started, or between phases)'
print('--- throttle events so far ---')
!grep -c 'governor. THROTTLE' v6_run.log 2>/dev/null || echo 0
print('--- last 30 log lines ---')
!tail -n 30 v6_run.log 2>/dev/null || echo '(no log yet)'

### A5c. Thread-calibration curve
What the sweep measured, per budget level. `SustainedTurnsPerMin` is the second of two timed segments —
that is the number the knee is chosen on. `ThrottlePct` is how much slower the second segment was than
the first: a few percent is warm-up noise, a large positive value means the machine could not hold the
clock, and `ClockDropPct` (Linux only) shows the same thing directly.

In [ ]:
import pandas as pd, os
p = 'test_results/v6/_calibration/thread_calibration.csv'
if os.path.exists(p):
    t = pd.read_csv(p)
    cols = ['BudgetUnits','Threads','SustainedTurnsPerMin','ThrottlePct','SpeedupVs1',
            'EfficiencyPct','ClockDropPct','PeakHeapMB','PerTaskHeapMB','Chosen']
    display(t[cols].round(2))
    print('chosen per level:')
    display(t[t.Chosen == 'yes'][['BudgetUnits','Threads','SustainedTurnsPerMin','PerTaskHeapMB']])
else:
    print('no thread_calibration.csv yet - the first level has not been swept')

### A5d. Stopping the run
Use **SIGTERM, not `kill -9`**. SIGTERM is a cooperative stop: queued turns are abandoned, running turns
finish and write their rows, and no config is marked complete unless it really is — so nothing is lost
and §A5 resumes exactly where it stopped. `kill -9` skips all of that and costs the turns that were in
flight (at most one per worker); the resume still recovers, it just repeats those.

The lock is released on a clean stop, and a lock left by a `kill -9` goes stale on its own within two
minutes — no manual cleanup either way.

In [ ]:
# Graceful stop. Re-run section A5 afterwards to continue.
!pkill -TERM -f RunExperimentV6 && echo 'SIGTERM sent - watch v6_run.log for the drain banner' || echo '(nothing running)'

## A6. Output integrity — run this before ANY analysis
Scans every leaf for the damage an interrupted or doubled run leaves: duplicated `(scenario, visit)`
turns, results rows with no telemetry row behind them, rows torn by a kill mid-write, and leaves marked
complete in `_completed.txt` that are not.

The duplicate case is the one to care about. Two writers on one tree — a backgrounded run that outlived
its kernel, plus a "resume" started beside it — produce a dataset with rows twice over. It does not
crash and it does not warn; it just doubles that turn's weight in every average downstream. The run
lock now prevents it, but this verifies the result, including for data produced before the lock existed.

`--repair` rewrites the affected leaves keeping the first complete copy of each turn (`.bak` kept), and
un-marks any leaf that is not actually finished so the next run completes it.

In [ ]:
!java -cp "out:lib/*" core.data.exp.VerifyV6Output test_results/v6

In [ ]:
# Only if the cell above reported problems:
# !java -cp "out:lib/*" core.data.exp.VerifyV6Output test_results/v6 --repair

## A7. Analyse (after the isolation run completes)
`analyze_v6_full.py` imports `analyze_v6_base` — the v5 analysis engine, vendored into v6 so nothing
here depends on a retired v5 file — and re-points it at the v6 tree, so **every v5 table and figure is
produced by identical code** (T1–T8, fig3–fig8, E1–E16, and the hypothesis battery E17–E21) — which is
what makes a v5-vs-v6 comparison meaningful. Scenario counts are read from the data, so the
smaller portfolio needs no change here. Two v6 data fixes are applied in the loader: `FmCalls` is
remapped to search-only calls (raw total kept as `FmCallsTotal`) so inherited cost metrics are not
inflated by the excluded greedy initialization, and the per-burn `ComputeUnitsElapsed` is dropped at
turn level so it cannot be silently averaged.

Then the new budget-adherence section:

| table | content |
|---|---|
| `B1_budget_adherence` | realized `SearchUnits` vs target per agent per level: mean, SD, overshoot, within-turn pacing |
| `B2_op_mix` | advances/copies/evals per turn, and what an equal-fmcalls budget *would* have given each agent |
| `B3_reference_conversion` | the C0 units→seconds table, split by config and by scenario source |
| `B4_init_cost` | `InitUnits` by source and BF tertile, and its share of the budget |

See `analysis/BUDGET_SUPPLEMENT.md` for the paper-facing write-up of what these establish.

In [ ]:
!python3 analysis/analyze_v6_full.py test_results/v6

## A8. Significance — the inferential answer
`analyze_v6_full.py` describes the data; this tests it, and it is what the write-up should quote.

Two things it does that the descriptive tables do not:

1. **Clusters by scenario.** The descriptive CI pools all ~5,000 turns per cell as if independent, but
   the 50 visits of a scenario share a board, a position and a greedy root. The independent replicate is
   the scenario (n=100). Pooling understates the standard error by up to sqrt(50) ~ 7x, at which point
   every difference looks significant. Visits are averaged first, then a paired Wilcoxon signed-rank
   with Benjamini-Hochberg correction, plus Cliff's delta so effect size is separate from significance.
2. **Reads the design as the 2x2 factorial it is.** `EMCTS` / `CrossoverBB` / `CrossoverExpansion@r` /
   `CrossoverCombo@r` are the four cells of BB-crossover x expansion-crossover. That identifies the
   **BB main effect**, the **expansion main effect**, and their **interaction** — whether the two
   mechanisms reinforce each other or get in each other's way. Comparing each config to EMCTS, which is
   all the descriptive analysis does, cannot separate those.

It also re-checks that each operator actually fired, so a cell where crossover never triggered is not
silently read as "crossover at this ratio does nothing".

In [ ]:
!python3 analysis/v6_significance.py test_results/v6
print(open('analysis/v6_significance/SIGNIFICANCE_V6.md').read())

---
# PART B — Full games (`test_results/v6_full`)

EMCTS plays complete games against every experimental configuration, at every ratio and every budget.
**The only thing measured is who won** — search telemetry is Part A's job, and this phase answers the
question Part A deliberately cannot.

**Do not start Part B while Part A is still running.** Check §0.1 first: `pgrep -af java` should be
empty and neither tree should hold a live lock.

Design: `analysis/EXPERIMENT_V6_FULL_GAMES_PLAN.md`. Machine placement: `analysis/V6_SERVER_RUNBOOK.md`.

| | |
|---|---|
| matchups | EMCTS vs 11 experimental configs × 3 budgets = **33** |
| budgets | 40,000 / 80,000 / 400,000 compute-units per turn (the same levels Part A uses) |
| games | 100 per matchup → **3,300 total** |
| maps | 25 fixed, each played 4 times: **2 per orientation** |
| expected cost | ~410–575 CPU-hours; the 400k level is ~90% of it |

**No EMCTS-vs-EMCTS control arm.** Earlier versions played one at each level and read its win rate as
the check that no seat or spawn asymmetry survived the orientation swap. That check is now made where
it is cheaper and sharper: `NewExperimentSmokeTestV6Full` §2 asserts the swap *directly* — identical
board, capitals exchanged, seats exchanged, on every map — `VerifyV6FullOutput` checks the realized
schedule, and W6 of the analysis reports every matchup's win rate by orientation and by seat, so a
residual bias would show in all 33 arms rather than in one. The 300 games that bought are spent on the
matchups that answer the actual question.

**Why 25 maps × 4 rather than 20 × 5.** At a fixed game count, the variance of a win rate is dominated
by the between-map term, so more maps beat more repetitions per map; and 5 games per map cannot be
split evenly between the two orientations, which the mirror-pair design requires. Maps 0–19 are the
same boards the 20-map set used — the seeds are taken in index order — so the set grew rather than
being replaced.

## B1. Smoke test (~1 min) — correctness gate

32 checks. The ones that matter most:

- **§2** the orientation swap is a real exchange — identical board, capitals swapped, seats swapped —
  asserted on all 25 maps. With no control arm downstream, this is the primary evidence that position
  and first-move advantage cancel;
- **§1c** capital positions and distances actually vary across the map set (this catches the
  `java.util.Random` cross-seed correlation that made every map geometrically identical in the first
  version);
- **§3** the 100-game schedule is balanced: every map played 4 times, 2 in each orientation, and
  (map, orientation, pair) unique per game;
- **§5h** self-play can report losses (both sides share a name, so a name-based winner would silently
  make every EMCTS row a win);
- **§6** the same games replay byte-identically across a thread pool;
- **§8** the runner restores the engine constants — a preceding thread calibration lowers
  `MAX_TURNS_CAPITALS` to its own 2–12 tick cap, and a leak there would play the whole tournament to
  the wrong cap with nothing in the output to say so.

A failure here means do not run the experiment.

In [ ]:
!java -Xmx4g -cp "out:lib/*" core.data.exp.NewExperimentSmokeTestV6Full

## B2. The map set (instant)

Generates the 25 maps and writes `test_results/v6_full/_maps/maps.csv` with capital positions, capital
distance and per-start terrain and resource counts.

**Asymmetry is measured, not filtered.** Every map is played 2 times in each orientation, so a map bias
of any size cancels exactly within that map; filtering would narrow the map distribution the result
generalises over while removing a bias the schedule already removes. The numbers are here so the
write-up can state what the maps looked like.

In [ ]:
!java -Xmx4g -cp "out:lib/*" core.data.exp.FullGameMapSetV6 test_results/v6_full/_maps

In [ ]:
import pandas as pd
m = pd.read_csv('test_results/v6_full/_maps/maps.csv')
print(f'{len(m)} maps, board {m.BoardSize.iloc[0]}x{m.BoardSize.iloc[0]}')
print('capital distance :', dict(m.CapitalDistance.value_counts().sort_index()))
print('start asymmetry  : min %d  median %.1f  max %d' % (
    m.StartAsymmetry.min(), m.StartAsymmetry.median(), m.StartAsymmetry.max()))
print('distinct capital-A tiles:', m.groupby(['CapAX','CapAY']).ngroups,
      ' capital-B tiles:', m.groupby(['CapBX','CapBY']).ngroups)
print()
print(f'If either count is 1, the level seeds are correlated and these are not really {len(m)} maps.')
m[['MapIndex','LevelSeed','CapAX','CapAY','SeatA','CapBX','CapBY','SeatB',
   'CapitalDistance','StartAsymmetry']]

## B3. Game-length probe — sets the turn cap

**Run this and read it before §B5.**

A CAPITALS game ends either by conquest (one tribe holds every capital) or by the turn cap, after which
the score ranking decides. Those measure different things: conquest is playing strength, and score at a
fixed horizon is dominated in Tribes by technology count. The old Phase 2 tournament capped at 50 and
only 1.3% of games reached it — but those agents planned on a 500–1000 ms clock and these are far
stronger, so the cap has to be re-measured rather than inherited.

The probe plays real games under a loose safety cap and recommends the smallest cap that would decide
at most 2% of games. Games that reach the safety cap are **censored, not dropped**. It samples one
matchup per family — CrossoverBB, CrossoverExpansion@5, CrossoverCombo@5 — because game length is a
property of the matchup rather than of a single agent.

Cost: minutes at 40k/80k, up to ~1–2 h at 400k. Args: `[threads|auto] [safetyCap] [gamesPerLevel]`.

In [ ]:
!nohup java -Xmx200g -XX:+UseG1GC -cp "out:lib/*" \
    core.data.exp.GameLengthProbeV6 auto 150 60,60,24 > v6full_probe.log 2>&1 &
print('started; re-run the next cell to watch it')

In [ ]:
!tail -40 v6full_probe.log

### B3b. Set the turn cap

Read the RECOMMENDATION block above. If it differs from the compiled `RunV6FullGames.TURN_LIMIT`, edit
that constant, re-run §0.2, and only then continue.

It is a compile-time constant on purpose: the tournament's rules should be fixed and visible in the
source, and a dataset must never silently contain games played under two different caps.

In [ ]:
import re, pathlib
src = pathlib.Path('src/core/data/exp/RunV6FullGames.java').read_text()
print('compiled TURN_LIMIT =', re.search(r'TURN_LIMIT\s*=\s*(\d+)', src).group(1))
!grep -A2 'RECOMMENDED TURN CAP' v6full_probe.log || echo '(probe has not finished)'

## B4. Thread calibration — per budget level

A separate sweep from Part A's, because the work item is different: a **game**, not a turn. It sweeps
thread counts on real (short) games and picks the smallest count within 3% of the best *sustained*
throughput — the second of two timed segments, so a point that only reaches peak on boost clocks shows
up as `ThrottlePct` rather than being chosen.

Its honest limitation: the games it times are short and therefore early-game, so its per-task heap
estimate is low. It applies a 2× derate before gating, and the live `LoadGovernor` during §B5 is the
real protection.

Skippable — §B5 calibrates any level with no `_tuned_threads_full_<cu>cu.txt` — but running it here
keeps the sweep out of the run's own log.

In [ ]:
!java -Xmx700g -XX:+UseG1GC -cp "out:lib/*" core.data.exp.ThreadCalibrationV6Full \
    test_results/v6_full/_calibration

In [ ]:
import pandas as pd, os
p = 'test_results/v6_full/_calibration/thread_calibration_full.csv'
if os.path.exists(p):
    c = pd.read_csv(p)
    display(c[['ComputeUnits','Threads','SustainedGamesPerMin','ThrottlePct','Speedup',
               'EfficiencyPct','ClockDropPct','PeakHeapMB','Chosen']])
    print()
    print('Chosen per level:')
    print(c[c.Chosen=='yes'][['ComputeUnits','Threads','SustainedGamesPerMin']].to_string(index=False))
else:
    print('not calibrated yet')

## B5. Run the tournament — and the RESUME cell

Launches detached so it survives a disconnect. **Re-run this same cell verbatim to resume:** finished
matchups are skipped whole, a partial matchup continues per game, and a directory lock refuses a second
writer over the same tree.

Every game of a level goes into one pool rather than a pool per matchup, so the workers stay busy and
no matchup's tail waits on its single longest game.

Check the banner line **`Compute profiles: 33/33 calibrated`** in the first lines of `v6full_run.log`.
The tournament reads the same `test_results/v6/_calibration/compute_weights.properties` that §0.5
wrote, so a fresh tree with no calibration prices every agent with the global weights — the budget is
then not equal compute between EMCTS and its opponent, which is the one thing this phase cannot
tolerate.

In [ ]:
!nohup java -Xmx750g -XX:+UseG1GC -XX:MaxGCPauseMillis=500 \
    -cp "out:lib/*" core.data.exp.RunV6FullGames auto > v6full_run.log 2>&1 &
print('started - watch with the progress cell below')

### B5b. Progress (re-run to refresh)

Read-only view over the output tree. Touches nothing the running job owns.

In [ ]:
import os, glob, pandas as pd
BASE = 'test_results/v6_full'
EXPECTED_PER_LEAF = 100          # FullGameRunnerV6.GAMES_PER_MATCHUP (25 maps x 4)
EXPECTED_LEAVES = 33             # 11 configs x 3 budget levels

rows = []
for p in sorted(glob.glob(f'{BASE}/**/games.csv', recursive=True)):
    leaf = os.path.relpath(os.path.dirname(p), BASE)
    try:
        d = pd.read_csv(p)
    except Exception:
        continue
    rows.append(dict(leaf=leaf, games=len(d), pct=round(100*len(d)/EXPECTED_PER_LEAF),
                     emcts_win=round(100*d.EmctsWon.mean(),1) if len(d) else None,
                     conquest=round(100*(d.DecidedBy=='CAPITAL').mean()) if len(d) else None))
prog = pd.DataFrame(rows)
if len(prog):
    total, expected = prog.games.sum(), EXPECTED_LEAVES*EXPECTED_PER_LEAF
    print(f'{total:,} / {expected:,} games  ({100*total/expected:.1f}%)')
    display(prog)
else:
    print('no games yet')

print('--- checkpointed matchups ---')
!wc -l < test_results/v6_full/_completed.txt 2>/dev/null || echo 0
print('--- live process ---')
!pgrep -af RunV6FullGames || echo '(not running)'
!ps -o rss=,nlwp= -p $(pgrep -f RunV6FullGames | head -1) 2>/dev/null | awk '{printf "RSS %.1f GB, %d threads\n", $1/1048576, $2}'
print('--- log tail ---')
!tail -12 v6full_run.log

### B5c. Stopping

**SIGTERM, never `kill -9`.** SIGTERM is a cooperative stop: queued games are abandoned, running games
finish and write their row, and no matchup is marked complete unless it really is. A hard kill can leave
a torn final row — recoverable by §B6, but there is no reason to create the work.

In [ ]:
!pkill -TERM -f RunV6FullGames && echo 'SIGTERM sent - watch v6full_run.log for the drain' || echo 'not running'

## B6. Output check — run before ANY analysis

Looks for what an interrupted or doubled run leaves (torn rows, duplicate GameIds) **and** for schedule
violations, which matter more: a duplicated row is obvious in a count, whereas a matchup that played one
orientation more often than the other looks like ordinary data and would bias the win rate by exactly
the amount the swap exists to remove. With no control arm to catch that after the fact, this check is
load-bearing.

It also prints the conquest share and the cap-bind rate.

In [ ]:
!java -cp "out:lib/*" core.data.exp.VerifyV6FullOutput test_results/v6_full

In [ ]:
# Only if the cell above reported fixable problems (duplicates / torn rows):
# !java -cp "out:lib/*" core.data.exp.VerifyV6FullOutput test_results/v6_full --repair

## B7. Analyse

Win rates with **map-clustered** intervals (4 games share a map, so a plain binomial interval
understates uncertainty), the pooled per-family headline test, the per-ratio curve, and the mirror-pair
sign test.

**Read W6 first** — the balance table. It reports each matchup's win rate by orientation and by seat,
and it is where a surviving position or turn-order bias would appear now that there is no self-play
control arm. A systematic gap in the same direction across all 33 arms means what a control off 50%
would have meant. W4 (the control table) is written for compatibility and is normally empty.

In [ ]:
!python3 analysis/analyze_v6_full_games.py test_results/v6_full --outdir analysis/v6_full_games

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('analysis/v6_full_games/RESULTS_V6_FULL_GAMES.md').read()))

In [ ]:
import pandas as pd
# W6: the balance diagnostic that replaces the control arm. OrientationGapPP is the number to scan.
w6 = pd.read_csv('analysis/v6_full_games/W6_balance.csv')
display(w6.sort_values('OrientationGapPP', ascending=False).head(12))
print('median orientation gap: %.1f pp   median seat-0 win rate: %.3f'
      % (w6.OrientationGapPP.median(), w6.Win_seat0.median()))

In [ ]:
from IPython.display import Image, display
import os
for f in ['F1_winrate_by_ratio.png', 'F2_forest.png']:
    p = f'analysis/v6_full_games/figures/{f}'
    if os.path.exists(p):
        display(Image(p))